# Stage 05 — Data Storage Homework


In [ ]:
# --- packages this notebook needs ---
# Uncomment and run once if needed, then re-comment.
#
# !pip install numpy
# !pip install pandas
# !pip install pyarrow
# !pip install python-dotenv


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.9/35.9 MB 34.4 MB/s  0:00:01m0:00:0100:01


## 1. Environment-driven paths

In [1]:
import os
import pathlib
import datetime as dt

import numpy as np
import pandas as pd
from dotenv import load_dotenv, set_key


# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------
# If the notebook is run from homework4/notebooks/, go up one level.
# If it is run from homework4/ directly, keep the current directory.
CWD = pathlib.Path.cwd()

if CWD.name == "notebooks":
    PROJECT_ROOT = CWD.parent
else:
    PROJECT_ROOT = CWD

print("Current working directory:", CWD.resolve())
print("PROJECT_ROOT:", PROJECT_ROOT.resolve())


# ------------------------------------------------------------
# Ensure .env has the two required storage variables.
# Existing unrelated .env entries are preserved.
# ------------------------------------------------------------
ENV_PATH = PROJECT_ROOT / ".env"
ENV_PATH.touch(exist_ok=True)

set_key(str(ENV_PATH), "DATA_DIR_RAW", "data/raw", quote_mode="never")
set_key(str(ENV_PATH), "DATA_DIR_PROCESSED", "data/processed", quote_mode="never")

load_dotenv(ENV_PATH, override=True)


def get_env_path(name: str, default: str) -> pathlib.Path:
    """Read a path from .env and resolve relative paths from PROJECT_ROOT."""
    path = pathlib.Path(os.getenv(name, default))
    return path if path.is_absolute() else PROJECT_ROOT / path


RAW_DIR = get_env_path("DATA_DIR_RAW", "data/raw")
PROC_DIR = get_env_path("DATA_DIR_PROCESSED", "data/processed")

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROC_DIR.mkdir(parents=True, exist_ok=True)

print("\n.env path:", ENV_PATH.resolve())
print("RAW_DIR:", RAW_DIR.resolve())
print("PROC_DIR:", PROC_DIR.resolve())


Current working directory: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/notebooks
PROJECT_ROOT: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5

.env path: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/.env
RAW_DIR: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/data/raw
PROC_DIR: /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/data/processed


## 2. Create the sample DataFrame

In [2]:
# Same sample-data pattern used in the lecture notebook.
np.random.seed(5)

dates = pd.date_range("2024-01-01", periods=10, freq="D")

df = pd.DataFrame({
    "date": dates,
    "ticker": ["AAPL"] * 10,
    "price": 150 + np.random.randn(10).cumsum(),
})

display(df)
df.info()


,date,ticker,price
0,2024-01-01,AAPL,150.441227
1,2024-01-02,AAPL,150.110357
2,2024-01-03,AAPL,152.541129
3,2024-01-04,AAPL,152.289036
4,2024-01-05,AAPL,152.398646
5,2024-01-06,AAPL,153.981127
6,2024-01-07,AAPL,153.071895
7,2024-01-08,AAPL,152.480258
8,2024-01-09,AAPL,152.667862
9,2024-01-10,AAPL,152.337992


<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    10 non-null     datetime64[us]
 1   ticker  10 non-null     str           
 2   price   10 non-null     float64       
dtypes: datetime64[us](1), float64(1), str(1)
memory usage: 412.0 bytes


## 3. Save in Two Formats

In [4]:
def ts():
    return dt.datetime.now().strftime("%Y%m%d-%H%M")


# Use one timestamp so the CSV and Parquet filenames clearly belong together.
run_ts = ts()

csv_path = RAW_DIR / f"prices_{run_ts}.csv"
parq_path = PROC_DIR / f"prices_{run_ts}.parquet"


# -------------------------
# CSV -> data/raw/
# -------------------------
df.to_csv(csv_path, index=False)
print("Saved CSV →", csv_path.resolve())


# -------------------------
# Parquet -> data/processed/
# -------------------------
parquet_saved = False

try:
    df.to_parquet(parq_path, index=False)
    parquet_saved = True
    print("Saved Parquet →", parq_path.resolve())

except (ImportError, ModuleNotFoundError) as e:
    print("\nParquet save failed because a Parquet engine is not installed.")
    print("Install one with: pip install pyarrow")
    print("Error:", e)

except Exception as e:
    print("\nParquet save failed.")
    print("A Parquet engine such as pyarrow or fastparquet is required.")
    print("Error:", e)


Saved CSV → /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/data/raw/prices_20260817-2040.csv
Saved Parquet → /Users/xuan/Desktop/bootcamp_Jiaxuan_Li/homework/homework5/data/processed/prices_20260817-2040.parquet


## 4. Reload and Validate

The validation function checks:

- DataFrame shape
- Required columns
- `date` remains datetime
- `ticker` remains string-like
- `price` remains numeric


In [5]:
def validate_loaded(
    original: pd.DataFrame,
    reloaded: pd.DataFrame,
    cols=("date", "ticker", "price"),
):
    """Validate shape, required columns, and important dtypes."""

    checks = {
        "shape_equal": original.shape == reloaded.shape,
        "cols_present": all(c in reloaded.columns for c in cols),
    }

    if "date" in reloaded.columns:
        checks["date_is_datetime"] = pd.api.types.is_datetime64_any_dtype(
            reloaded["date"]
        )

    if "ticker" in reloaded.columns:
        checks["ticker_is_string"] = (
            pd.api.types.is_string_dtype(reloaded["ticker"])
            or pd.api.types.is_object_dtype(reloaded["ticker"])
        )

    if "price" in reloaded.columns:
        checks["price_is_numeric"] = pd.api.types.is_numeric_dtype(
            reloaded["price"]
        )

    checks["all_passed"] = all(checks.values())
    return checks


In [6]:
# -------------------------
# Reload CSV
# -------------------------
df_csv = pd.read_csv(csv_path, parse_dates=["date"])

csv_validation = validate_loaded(df, df_csv)

print("CSV validation:")
print(csv_validation)


# -------------------------
# Reload Parquet
# -------------------------
if parq_path.exists():
    try:
        df_parq = pd.read_parquet(parq_path)

        parquet_validation = validate_loaded(df, df_parq)

        print("\nParquet validation:")
        print(parquet_validation)

    except Exception as e:
        print("\nParquet read failed.")
        print("Install a Parquet engine such as pyarrow.")
        print("Error:", e)
else:
    print("\nParquet file is not present because the earlier Parquet write was skipped.")


CSV validation:
{'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'all_passed': True}

Parquet validation:
{'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'all_passed': True}


## 5. IO Utilities (suffix-based)

In [7]:
from typing import Union


def ensure_dir(path: pathlib.Path):
    """Create a file's parent directory when it does not already exist."""
    path.parent.mkdir(parents=True, exist_ok=True)


def detect_format(path: Union[str, pathlib.Path]):
    """Determine storage format from the filename suffix."""
    suffix = pathlib.Path(path).suffix.lower()

    if suffix == ".csv":
        return "csv"

    if suffix in {".parquet", ".pq", ".parq"}:
        return "parquet"

    raise ValueError(
        f"Unsupported format for: {path}. "
        "Supported suffixes are .csv, .parquet, .pq, and .parq."
    )


def write_df(df: pd.DataFrame, path: Union[str, pathlib.Path]):
    """Write a DataFrame as CSV or Parquet based on its filename suffix."""

    path = pathlib.Path(path)
    ensure_dir(path)

    fmt = detect_format(path)

    if fmt == "csv":
        df.to_csv(path, index=False)

    elif fmt == "parquet":
        try:
            df.to_parquet(path, index=False)

        except (ImportError, ModuleNotFoundError) as e:
            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

        except Exception as e:
            raise RuntimeError(
                "Could not write Parquet file. "
                "Check that pyarrow or fastparquet is installed."
            ) from e

    return path


def read_df(path: Union[str, pathlib.Path]):
    """Read CSV or Parquet based on the filename suffix."""

    path = pathlib.Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    fmt = detect_format(path)

    if fmt == "csv":
        # Read the header first so we only parse `date` when it exists.
        header = pd.read_csv(path, nrows=0)

        if "date" in header.columns:
            return pd.read_csv(path, parse_dates=["date"])

        return pd.read_csv(path)

    elif fmt == "parquet":
        try:
            return pd.read_parquet(path)

        except (ImportError, ModuleNotFoundError) as e:
            raise RuntimeError(
                "Parquet engine not available. "
                "Install pyarrow or fastparquet."
            ) from e

        except Exception as e:
            raise RuntimeError(
                "Could not read Parquet file. "
                "Check that pyarrow or fastparquet is installed."
            ) from e


## 6. Test the Utility Functions

In [10]:
# Test write_df and read_df using the existing Task 1 paths

# CSV
write_df(df, csv_path)
df_csv_util = read_df(csv_path)

print("CSV utility validation:")
print(validate_loaded(df, df_csv_util))


# Parquet
try:
    write_df(df, parq_path)
    df_parquet_util = read_df(parq_path)

    print("\nParquet utility validation:")
    print(validate_loaded(df, df_parquet_util))

except RuntimeError as e:
    print("\nParquet utility test failed:")
    print(e)

CSV utility validation:
{'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'all_passed': True}

Parquet utility validation:
{'shape_equal': True, 'cols_present': True, 'date_is_datetime': True, 'ticker_is_string': True, 'price_is_numeric': True, 'all_passed': True}
